# Tahap 3: Pemodelan Reinforcement Learning (RL)
### Agen 1: Penjadwal Pemeliharaan Preventif Dinamis (Predictive Maintenance Scheduler)

Notebook ini membangun dan melatih agen Reinforcement Learning untuk meminimalkan total biaya operasional robot akibat downtime dan perbaikan. Kita akan:
1. **Membangun Custom Gymnasium Environment** bernama `WarehouseMaintenanceEnv` yang mensimulasikan keausan robot harian, risiko kerusakan (Weibull failure rate), antrean perbaikan, sertifikasi teknisi, dan kegagalan inspeksi.
2. **Melatih Agen RL** menggunakan algoritma **Tabular Q-Learning** dengan melakukan diskritisasi pada state space. Metode ini transparan, cepat konvergen, dan ideal untuk masalah penjadwalan pemeliharaan.
3. **Membandingkan Kebijakan RL** dengan kebijakan heuristik standar bisnis:
   * *Run-to-Failure (Corrective)*: Perbaiki hanya jika robot benar-benar rusak.
   * *Time-Based (Heuristic)*: Jadwalkan servis rutin setiap kali robot mencapai 3.500 jam operasional.
4. **Memvisualisasikan Akumulasi Biaya** untuk membuktikan keunggulan agen RL.

## 1. Import Library & Setup

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces

# Set seed untuk reproduksibilitas
np.random.seed(42)
random.seed(42)

# Non-interactive plotting backend untuk kelancaran running
import matplotlib
matplotlib.use('Agg')

print("Library berhasil di-load!")

Library berhasil di-load!


c:\Users\hpvic\AppData\Local\Programs\Python\Python310\lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment FetchSlide-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\hpvic\AppData\Local\Programs\Python\Python310\lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment FetchSlide-v3 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\hpvic\AppData\Local\Programs\Python\Python310\lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment FetchPickAndPlace-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\hpvic\AppData\Local\Programs\Python\Python310\lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment FetchPickAndPlace-v3 already in registry.
  logger.warn(

## 2. Merancang Custom Gymnasium Environment
Kita merancang environment di mana:
* **State (Observation)**:
  1. `operational_hours` (0 - 15,000 jam): Akumulasi jam operasi sejak perawatan terakhir.
  2. `recurrence_count` (0 - 7 kali): Berapa kali robot ini mengalami kerusakan berulang.
  3. `failure_risk` (0.0 - 1.0): Risiko kegagalan teoritis dihitung menggunakan fungsi Weibull: $P(fail) = 1 - e^{-(\text{hours}/6000)^2}$.
* **Actions**:
  * `0`: Do Nothing (Operasikan robot terus).
  * `1`: Jadwalkan Preventive Maintenance oleh Teknisi **Level-1** (Biaya murah, durasi sedang, risiko inspeksi gagal 25%).
  * `2`: Jadwalkan Preventive Maintenance oleh Teknisi **Master Tech** (Biaya mahal, durasi cepat, risiko inspeksi gagal 2%).
* **Reward (Cost-centric)**:
  * Penalti downtime: $-\$100$ per jam downtime (kerugian produktivitas bisnis).
  * Biaya servis: $-\text{labor cost} - \text{parts cost}$.
  * Penalti breakdown: $-\$1000$ jika robot rusak total (corrective breakdown).

In [2]:
class WarehouseMaintenanceEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self):
        super(WarehouseMaintenanceEnv, self).__init__()

        # Action Space: 0: Do Nothing, 1: PM Level-1 Tech, 2: PM Master Tech
        self.action_space = spaces.Discrete(3)

        # Observation Space: [operational_hours, recurrence_count, failure_risk]
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([15000.0, 10.0, 1.0], dtype=np.float32),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.operational_hours = 0.0
        self.recurrence_count = 0.0
        self.current_day = 0
        self.state = np.array([self.operational_hours, self.recurrence_count, self.get_failure_risk()], dtype=np.float32)
        return self.state, {}

    def get_failure_risk(self):
        # Distribusi Weibull untuk pemodelan keausan mesin (wear-out phase)
        # Lambda (karakteristik umur) = 6000 jam, k (shape parameter) = 2.0
        return float(1.0 - np.exp(- (self.operational_hours / 6000.0) ** 2.0))

    def step(self, action):
        self.current_day += 1
        reward = 0.0
        downtime = 0.0
        repair_cost = 0.0
        breakdown_penalty = 0.0
        is_broken = False

        if action == 0:  # Do Nothing / Run
            self.operational_hours += 24.0
            risk = self.get_failure_risk()
            if random.random() < (risk * 0.08):  # Skala harian risiko kegagalan
                is_broken = True
                downtime = random.uniform(8.0, 16.0)
                repair_cost = 1000.0  # Biaya perbaikan darurat
                breakdown_penalty = 1000.0  # Kerugian operasional ekstra
                self.operational_hours = 0.0  # Reset setelah diperbaiki
                self.recurrence_count += 1.0
                
        elif action == 1:  # Preventive Maintenance (Level-1 Tech)
            downtime = random.uniform(4.0, 8.0)
            # Upah Level-1 Tech = $50/jam + biaya parts $150
            repair_cost = (downtime * 50.0) + 150.0
            
            # 75% Lolos inspeksi
            if random.random() < 0.75:
                self.operational_hours = 0.0
                self.recurrence_count = 0.0
            else:
                self.operational_hours = self.operational_hours * 0.3
                self.recurrence_count += 1.0
                reward -= 200.0  # Penalti kegagalan inspeksi
                
        elif action == 2:  # Preventive Maintenance (Master Tech)
            downtime = random.uniform(2.0, 4.0)
            # Upah Master Tech = $100/jam + biaya parts $200
            repair_cost = (downtime * 100.0) + 200.0
            
            # 98% Lolos inspeksi
            if random.random() < 0.98:
                self.operational_hours = 0.0
                self.recurrence_count = 0.0
            else:
                self.operational_hours = self.operational_hours * 0.1
                self.recurrence_count += 1.0
                reward -= 100.0

        # Hitung Total Cost
        downtime_loss = downtime * 100.0
        total_cost = downtime_loss + repair_cost + breakdown_penalty
        reward -= total_cost

        # Update state
        self.state = np.array([self.operational_hours, self.recurrence_count, self.get_failure_risk()], dtype=np.float32)
        terminated = self.current_day >= 365
        truncated = False
        
        info = {"is_broken": is_broken, "cost": total_cost, "downtime": downtime}
        return self.state, reward, terminated, truncated, info

env = WarehouseMaintenanceEnv()
print("Environment divalidasi dengan sukses!")

Environment divalidasi dengan sukses!


## 3. Implementasi & Training Tabular Q-Learning
Kita membagi operational_hours ke dalam 10 bin (0, 1000, 2000, ..., 9000+) dan recurrence_count ke dalam 3 bin (0, 1, 2+), menghasilkan 30 state unik.

In [3]:
def get_state_id(state):
    h_bin = min(int(state[0] // 1000), 9)
    r_bin = min(int(state[1]), 2)
    return h_bin * 3 + r_bin

# Inisialisasi Q-Table (30 States x 3 Actions)
q_table = np.zeros((30, 3))

# Hyperparameter Q-Learning
alpha = 0.15          # Learning rate
gamma = 0.90          # Discount factor
epsilon = 1.0         # Exploration rate awal
epsilon_decay = 0.9995
min_epsilon = 0.01
episodes = 12000

print("Memulai training Q-Learning agent...")

for episode in range(episodes):
    state, _ = env.reset()
    sid = get_state_id(state)
    done = False
    
    while not done:
        # Epsilon-Greedy action selection
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[sid])
            
        n_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        nsid = get_state_id(n_state)
        
        # Bellman Equation Update
        q_table[sid, action] += alpha * (reward + gamma * np.max(q_table[nsid]) - q_table[sid, action])
        sid = nsid
        
    # Decay epsilon
    epsilon = max(min_epsilon, epsilon * epsilon_decay)

print("Training selesai!")

Memulai training Q-Learning agent...
Training selesai!


## 4. Evaluasi & Simulasi Perbandingan Kebijakan
Mari kita uji performa agen RL kita dengan 2 kebijakan standar industri lainnya selama periode 365 hari:
1. **Run-To-Failure Policy**: Hanya melakukan tindakan pemeliharaan jika robot rusak (mengambil tindakan perbaikan darurat).
2. **Time-Based Heuristic (Threshold) Policy**: Melakukan perawatan preventif oleh Master Tech setiap kali jam operasional robot mencapai 3.500 jam ($>3500$).
3. **Trained RL Policy (Q-Learning)**: Keputusan diambil berdasarkan Q-Table yang telah dilatih.

In [4]:
def simulate_policy(env, policy_type, q_table=None):
    # Gunakan seed tetap untuk evaluasi yang adil
    random.seed(100)
    np.random.seed(100)
    
    state, _ = env.reset()
    done = False
    total_cost = 0.0
    cumulative_costs = []
    actions_taken = []
    breakdowns = 0
    
    while not done:
        obs_hours = state[0]
        sid = get_state_id(state)
        
        if policy_type == 'run_to_failure':
            action = 0  # Do Nothing
        elif policy_type == 'time_based':
            if obs_hours > 3500.0:
                action = 2  # Jadwalkan PM Master Tech
            else:
                action = 0
        elif policy_type == 'rl':
            action = np.argmax(q_table[sid])
            
        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        cost = info['cost']
        total_cost += cost
        cumulative_costs.append(total_cost)
        actions_taken.append(action)
        if info['is_broken']:
            breakdowns += 1
            
    return cumulative_costs, actions_taken, breakdowns

# Run Simulasi
eval_env = WarehouseMaintenanceEnv()

costs_rtf, acts_rtf, breaks_rtf = simulate_policy(eval_env, 'run_to_failure')
costs_time, acts_time, breaks_time = simulate_policy(eval_env, 'time_based')
costs_rl, acts_rl, breaks_rl = simulate_policy(eval_env, 'rl', q_table=q_table)

print("=== HASIL SIMULASI 365 HARI ===")
print(f"Run-To-Failure : Total Cost = ${costs_rtf[-1]:,.2f} | Total Rusak = {breaks_rtf} kali")
print(f"Time-Based PM  : Total Cost = ${costs_time[-1]:,.2f} | Total Rusak = {breaks_time} kali")
print(f"RL Agent (Q-L) : Total Cost = ${costs_rl[-1]:,.2f} | Total Rusak = {breaks_rl} kali")

# Hitung Persentase Penghematan
savings_vs_rtf = ((costs_rtf[-1] - costs_rl[-1]) / costs_rtf[-1]) * 100
savings_vs_time = ((costs_time[-1] - costs_rl[-1]) / costs_time[-1]) * 100
print("\nRL Agent menghemat {:.1f}% dibanding Run-to-Failure".format(savings_vs_rtf))
print("RL Agent menghemat {:.1f}% dibanding Time-Based Heuristic".format(savings_vs_time))

=== HASIL SIMULASI 365 HARI ===
Run-To-Failure : Total Cost = $5,842.19 | Total Rusak = 2 kali
Time-Based PM  : Total Cost = $5,842.19 | Total Rusak = 2 kali
RL Agent (Q-L) : Total Cost = $5,835.72 | Total Rusak = 1 kali

RL Agent menghemat 0.1% dibanding Run-to-Failure
RL Agent menghemat 0.1% dibanding Time-Based Heuristic


## 5. Visualisasi Perbandingan Akumulasi Biaya

In [5]:
plt.figure(figsize=(12, 7))
plt.plot(costs_rtf, label=f'Run-To-Failure (Total Cost: ${costs_rtf[-1]:,.0f}, Rusak: {breaks_rtf}x)', color='crimson', linestyle='--', linewidth=2)
plt.plot(costs_time, label=f'Time-Based PM (>3500h) (Total Cost: ${costs_time[-1]:,.0f}, Rusak: {breaks_time}x)', color='darkorange', linestyle='-.', linewidth=2)
plt.plot(costs_rl, label=f'Trained RL Agent (Q-Learning) (Total Cost: ${costs_rl[-1]:,.0f}, Rusak: {breaks_rl}x)', color='forestgreen', linewidth=3)

plt.title("Analisis Perbandingan Akumulasi Biaya Operasional & Pemeliharaan (Simulasi 1 Tahun)", fontsize=13, pad=15)
plt.xlabel("Hari dalam Setahun (1 - 365)", fontsize=11)
plt.ylabel("Akumulasi Biaya Operasional (USD)", fontsize=11)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()

# Simpan plot untuk proposal bisnis
plt.savefig('docs/plots/maintenance_policy_comparison.png', bbox_inches='tight')
plt.show()

C:\Users\hpvic\AppData\Local\Temp\ipykernel_14432\4171139774.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Analisis Hasil Kebijakan RL
Mari kita amati pola tindakan (*action*) yang diambil oleh agen RL untuk memahami strategi optimal yang dipelajarinya:

In [6]:
# Analisis frekuensi aksi yang dipilih oleh agen RL
actions_rl_series = pd.Series(acts_rl)
action_counts = actions_rl_series.value_counts().rename({
    0: 'Do Nothing / Run',
    1: 'PM Level-1 Tech',
    2: 'PM Master Tech'
})

print("=== Tindakan yang Diambil oleh Agen RL Sepanjang Tahun ===")
print(action_counts)

# Temukan kapan agen melakukan pemeliharaan rutin
maintenance_days = [i+1 for i, x in enumerate(acts_rl) if x in [1, 2]]
print(f"\nRL memutuskan melakukan perawatan preventif pada hari-hari ke: {maintenance_days}")

=== Tindakan yang Diambil oleh Agen RL Sepanjang Tahun ===
Do Nothing / Run    362
PM Master Tech        3
Name: count, dtype: int64

RL memutuskan melakukan perawatan preventif pada hari-hari ke: [85, 199, 284]
